In [ ]:
import netCDF4 as nc
import os

path = "/mnt/backup_ETH/Marina/MERRA2/daily_fields"
files = [f for f in os.listdir(path) if f.endswith('.nc')]
files.sort()

print(f"{'Filename':<50} | {'Variables':<25} | {'Dimensions':<20} | {'Time Range'}")
print("-" * 120)

for file in files:
    try:
        f_path = os.path.join(path, file)
        ds = nc.Dataset(f_path)
        
        # 获取所有非坐标变量
        vars_list = [v for v in ds.variables if v not in ['time', 'lat', 'lon', 'lev', 'bnds']]
        vars_str = ", ".join(vars_list)
        
        # 获取维度大小
        dims = ds.dimensions
        dim_str = " x ".join([f"{k}({len(v)})" for k, v in dims.items() if k != 'bnds'])
        
        # 尝试提取时间跨度 (假设单位是 minutes/hours since...)
        time_var = ds.variables.get('time')
        if time_var and len(time_var) > 0:
            import netCDF4
            times = netCDF4.num2date(time_var[[0, -1]], time_var.units, time_var.calendar)
            time_range = f"{times[0].strftime('%Y%m%d')}-{times[1].strftime('%Y%m%d')}"
        else:
            time_range = "N/A"
            
        print(f"{file:<50} | {vars_str:<25} | {dim_str:<20} | {time_range}")
        ds.close()
    except Exception as e:
        print(f"{file:<50} | Error: {e}")

In [ ]:
import os
import re
from collections import defaultdict

path = "/pool/datos/reanalisis/merra2/daily/zm"
files = [f for f in os.listdir(path) if f.endswith('.nc')]

# 匹配模式：MERRA2_day_ (变量名) _ (时间).nc
# 模式说明：提取 MERRA2_day_ 之后到最后倒数第二个下划线之前的部分作为变量名
pattern = re.compile(r"MERRA2_day_(.*)_(\d{6})\.nc")

stats = defaultdict(list)

for f in files:
    match = pattern.match(f)
    if match:
        var_name = match.group(1)
        time_str = match.group(2)
        stats[var_name].append(time_str)

print(f"{'Variable Identifier':<20} | {'Count':<6} | {'Start':<8} | {'End':<8} | {'Example File'}")
print("-" * 100)

for var in sorted(stats.keys()):
    times = sorted(stats[var])
    start = times[0]
    end = times[-1]
    count = len(times)
    # 构造该变量的一个示例文件名
    example_file = f"MERRA2_day_{var}_{start}.nc"
    
    print(f"{var:<20} | {count:<6} | {start:<8} | {end:<8} | {example_file}")

In [ ]:
import netCDF4 as nc
import os

path = "/pool/datos/reanalisis/merra2/daily/zm"
# 刚才脚本输出的示例文件列表
example_files = [
    "MERRA2_day_DUDTGWD_zm_198001.nc", "MERRA2_day_epfy_198001.nc", 
    "MERRA2_day_epfz_198001.nc", "MERRA2_day_psitem_198001.nc",
    "MERRA2_day_t_zm_198001.nc", "MERRA2_day_u_zm_198001.nc",
    "MERRA2_day_utendepfd_198001.nc", "MERRA2_day_utendvtem_198001.nc",
    "MERRA2_day_utendwtem_198001.nc", "MERRA2_day_v_zm_198001.nc",
    "MERRA2_day_vtem_198001.nc", "MERRA2_day_w_zm_198001.nc",
    "MERRA2_day_wtem_198001.nc"
]

print(f"{'Variable':<15} | {'Long Name':<40} | {'Units':<15} | {'Dimensions'}")
print("-" * 100)

for f_name in example_files:
    f_path = os.path.join(path, f_name)
    if not os.path.exists(f_path):
        continue
        
    try:
        ds = nc.Dataset(f_path)
        # 排除辅助变量，寻找主变量
        # 逻辑：寻找维度最多的变量，或者文件名中包含的变量名
        main_var = None
        for v_name in ds.variables:
            if v_name.lower() in f_name.lower():
                main_var = v_name
                break
        
        if not main_var: # 如果没找到，取最后一个非坐标变量
            coords = ['time', 'lat', 'lon', 'lev', 'pressure', 'latitude', 'longitude', 'bnds']
            potential_vars = [v for v in ds.variables if v not in coords]
            main_var = potential_vars[0] if potential_vars else None

        if main_var:
            v = ds.variables[main_var]
            long_name = getattr(v, 'long_name', 'N/A')
            units = getattr(v, 'units', 'N/A')
            # 格式化维度信息，例如 (lev: 42, lat: 181)
            dim_info = ", ".join([f"{d}:{len(ds.dimensions[d])}" for d in v.dimensions])
            print(f"{main_var:<15} | {long_name[:40]:<40} | {units:<15} | ({dim_info})")
        
        ds.close()
    except Exception as e:
        print(f"{f_name:<15} | Error: {e}")